In [0]:
from pyspark.sql.types import StructType, StructField, DoubleType, LongType, IntegerType

# 1. Define explicit schema with Class as IntegerType
txn_schema = StructType([
    StructField("Time", LongType(), True),
    *[StructField(f"V{i}", DoubleType(), True) for i in range(1, 29)],
    StructField("Amount", DoubleType(), True),
    StructField("Class", IntegerType(), True) # Optimized for 0 and 1
])

In [0]:
bronze_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .schema(txn_schema)
    .load("/Volumes/realtime_fraud_detection/00_landing/raw_data/transactions/")
)

In [0]:
query = (
    bronze_stream_df.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", "/Volumes/realtime_fraud_detection/00_landing/raw_data/transactions/_checkpoints")
    .table("realtime_fraud_detection.01_bronze.raw_transactions")
)

print("🚀 Bronze Stream Ingestion Started Successfully!")

In [0]:
display(spark.read.table("realtime_fraud_detection.01_bronze.raw_transactions"))